# AI-Face Phase E: Radial FFT + MLP
Attach `nguyentrann0703/aiface-pilot-32k-single-bin`. This notebook clones FaceTrace from GitHub main, like notebook 10. Enable Internet and GPU. No HF_TOKEN or code dataset is needed. ZIP source is an optional fallback.

Run once with RUN_FULL=False for unit tests and smoke. After it passes, set RUN_FULL=True and Save Version -> Save & Run All for the scientific run. Smoke outputs are saved separately.

Forensic-only is independent. Fusion requires the other person's **full AI-Face** baseline artifacts, including validation predictions OR feature_cache.pt + checkpoints, plus test predictions and run_metadata.json. Old Who Is AI outputs cannot be used.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, shutil, zipfile

SOURCE_MODE = 'git'  # Clone GitHub main; optional 'zip' for a source package
SOURCE_ZIP = None  # Optional exact path under /kaggle/input
REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git'
REPO_REF = 'main'  # For git mode, use the branch/tag containing spectral files
REPO_DIR = Path('/kaggle/working/FaceTrace')
RUN_FULL = False
DATASET_VERSION = None  # Fill the version shown by your attached dataset before the final run
DATASET_INPUT = None  # Optional exact dataset directory; otherwise auto-detect the .bin
BASELINE_DIR = None  # Optional unpacked artifact root containing global_only/local_only/global_local
OUT = Path('/kaggle/working/outputs/aiface_phase_e')
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')


In [ ]:
if SOURCE_MODE == 'zip':
    candidates = [Path(SOURCE_ZIP)] if SOURCE_ZIP else sorted(Path('/kaggle/input').rglob('facetrace_spectral_source.zip'))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the code ZIP dataset or set SOURCE_ZIP. Found: {candidates}')
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError('Source directory already exists. Restart the Kaggle session before loading a new source package.')
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(candidates[0]) as archive:
        for member in archive.infolist():
            target = (REPO_DIR / member.filename).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError(f'Unsafe source ZIP member: {member.filename}')
        archive.extractall(REPO_DIR)
elif SOURCE_MODE == 'git':
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError('Restart the session to clone a fresh source version.')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    raise ValueError('SOURCE_MODE must be zip or git')
assert (REPO_DIR / 'scripts/train_spectral.py').is_file(), 'Source does not contain the spectral implementation'
sys.path.insert(0, str(REPO_DIR))
print('Source ready:', REPO_DIR)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML>=6', 'pandas>=2.2', 'scikit-learn>=1.4', 'Pillow>=10', 'tqdm>=4.66'], check=True)
import torch, torchvision, yaml
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if RUN_FULL and not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Notebook Settings -> Accelerator before the full run.')
config = yaml.safe_load((REPO_DIR / 'configs/spectral.yaml').read_text())
config['experiment']['dataset_version'] = DATASET_VERSION
CONFIG_PATH = OUT / 'spectral_run.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
if RUN_FULL and DATASET_VERSION is None:
    raise ValueError('Set DATASET_VERSION to the attached dataset version before a final run.')


In [ ]:
def run_logged(command, log_name):
    with (OUT / log_name).open('w') as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line); log.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'Command failed ({code}); see {OUT / log_name}')

run_logged([sys.executable, '-u', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_spectral.py', '-v'], 'unit_tests.log')
command = [sys.executable, '-u', str(REPO_DIR / 'scripts/run_aiface_spectral_kaggle.py'), '--config', str(CONFIG_PATH), '--output-dir', str(OUT / 'forensic_only')]
if DATASET_INPUT:
    command.extend(['--input-root', str(DATASET_INPUT)])
run_logged(command + ['--smoke'], 'smoke.log')


In [ ]:
if RUN_FULL:
    run_logged(command, 'full_train.log')
else:
    print('Smoke finished. Set RUN_FULL=True, fill DATASET_VERSION, then Save Version -> Save & Run All.')


In [ ]:
import pandas as pd
from IPython.display import display
RESULT_DIR = OUT / ('forensic_only' if RUN_FULL else 'forensic_only_smoke')
print('SCIENTIFIC FULL RUN' if RUN_FULL else 'SMOKE ONLY: these metrics are not research results')
display(pd.read_csv(RESULT_DIR / 'train_log.csv'))
display(pd.read_csv(RESULT_DIR / 'test_metrics_by_generator.csv'))
display(json.loads((RESULT_DIR / 'test_metrics_overall.json').read_text()))
print('Predictions:', RESULT_DIR / 'predictions_test.csv')
print('Checkpoint:', RESULT_DIR / 'checkpoint.pt')


In [ ]:
if RUN_FULL and BASELINE_DIR:
    run_logged([sys.executable, '-u', str(REPO_DIR / 'scripts/fuse_aiface_spectral.py'), '--forensic-dir', str(RESULT_DIR), '--baseline-dir', str(BASELINE_DIR), '--output-dir', str(OUT)], 'fusion.log')
    for name in ['local_forensic', 'global_local_forensic']:
        print(name)
        display(pd.read_csv(OUT / name / 'test_metrics_by_generator.csv'))
        display(pd.read_csv(OUT / name / 'rescue_harm.csv'))
else:
    print('Fusion pending until full AI-Face Global/Local artifacts arrive. Forensic-only can run independently.')


In [ ]:
# Include source and logs for handoff; extracted images live in /kaggle/temp and are excluded.
snapshot = OUT / 'source'
for directory in ['src', 'scripts', 'configs', 'tests']:
    shutil.copytree(REPO_DIR / directory, snapshot / directory, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
archive = shutil.make_archive('/kaggle/working/facetrace_spectral_artifacts', 'zip', root_dir=OUT)
print('Download:', archive)
print('Keep the Saved Version URL and attached dataset version in your handoff.')
